# 注意力评分函数

query怎么去和key比较才可以得到一个注意力权重

10.2中使用$- \frac{1}{2} (x-x_i)^2$ 即距离越近，评分越高

这里的流程是：

$query \rightarrow score(k_i, q) \rightarrow softmax \rightarrow \alpha_i \rightarrow attention(q) = \sum \alpha_i v_i$

注意力汇聚函数为值的加权和：

$$f(\mathbf{q}, (\mathbf{k}_1, \mathbf{v}_1), \dots, (\mathbf{k}_m, \mathbf{v}_m)) = \sum_{i=1}^m \alpha(\mathbf{q}, \mathbf{k}_i)\mathbf{v}_i \in \mathbb{R}^v $$

* $\mathbf{q} \in \mathbb{R}^q$ 查询向量
* $(\mathbf{k}_i, \mathbf{v}_i)$ 键值对
    * $\mathbf{k}_i \in \mathbb{R}^k$ 键向量：每个候选信息的特征标识，用于与查询 $\mathbf{q}$ 进行匹配比较
    * $\mathbf{v}_i \in \mathbb{R}^v$ 值向量：真正要提取的信息本身

* $\alpha(\mathbf{q}, \mathbf{k}_i) \in \mathbb{R}$： 表示查询 $\mathbf{q}$ 对第 $i$ 个键 $\mathbf{k}_i$ 的分配程度

$\alpha(\mathbf{q}, \mathbf{k}_i) \in \mathbb{R}$ 需要经过softmax运算得到

$$\alpha(\mathbf{q}, \mathbf{k}_i) = \mathrm{softmax}(a(\mathbf{q}, \mathbf{k}_i)) = \frac{\exp(a(\mathbf{q}, \mathbf{k}_i))}{\sum_{j=1}^m \exp(a(\mathbf{q}, \mathbf{k}_j))} \in \mathbb{R} $$

不同的评分函数a会导致不同的注意力汇聚操作

## 掩蔽softmax操作

相比于普通的注意力权重公式， 超出有效长度的位置都将被设置为0

In [1]:
import math
import torch
from torch import nn
from d2l import torch as d2l

#@save
def masked_softmax(X, valid_lens):
    """通过在最后一个轴上掩蔽元素来执行softmax操作"""
    # X:3D张量 （batch_size, query_num, key_num)
    # valid_lens:1D或2D张量,表示多少个key是有效的
    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)
    else:
        shape = X.shape
        if valid_lens.dim() == 1:  # 比如valid_lens=[2, 3] 表示第一个query的有效长度是2，第二个query的有效长度是3
            valid_lens = torch.repeat_interleave(valid_lens, shape[1]) # 重复广播到每个query
        else:
            valid_lens = valid_lens.reshape(-1)  # 拉成1D张量
        # 最后一轴上被掩蔽的元素使用一个非常大的负值替换，从而其softmax输出为0
        X = d2l.sequence_mask(X.reshape(-1, shape[-1]), valid_lens,
                              value=-1e6)  # （batch_size * query_num, key_num） valid_lens对应一个query的张量
# 把无效位置的value变为一个巨大的负数，从而使得其softmax输出为0
        return nn.functional.softmax(X.reshape(shape), dim=-1)  # 恢复形状之后再对最后一维做softmax

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


测试一下：

In [2]:
masked_softmax(torch.rand(2, 2, 4), torch.tensor([2, 3]))

tensor([[[0.6114, 0.3886, 0.0000, 0.0000],
         [0.4910, 0.5090, 0.0000, 0.0000]],

        [[0.3092, 0.3945, 0.2963, 0.0000],
         [0.2020, 0.3933, 0.4047, 0.0000]]])

## 加性注意力

query和key是不同长度的矢量的时候，可以使用加性注意力作为评分函数

$$a(\mathbf{q}, \mathbf{k}) = \mathbf{w}_v^\top \tanh(\mathbf{W}_q \mathbf{q} + \mathbf{W}_k \mathbf{k}) \in \mathbb{R} $$

可学习的三个参数：
* $\mathbf{W}_q \in \mathbb{R}^{h \times q}$：查询的线性变换矩阵，将 $\mathbf{q}$ 映射到隐藏空间 $\mathbb{R}^h$。
* $\mathbf{W}_k \in \mathbb{R}^{h \times k}$：键的线性变换矩阵，将 $\mathbf{k}$ 映射到相同的隐藏空间 $\mathbb{R}^h$
* $\mathbf{w}_v \in \mathbb{R}^h$：输出层的权重向量，通过内积将 $h$ 维隐藏特征压缩为一个标量分值

使用tanh作为激活函数并且禁用偏置项

In [3]:
class AdditiveAttention(nn.Module):
    """加性注意力"""
    def __init__(self, key_size, query_size, num_hiddens, dropout, **kwargs):
        super(AdditiveAttention, self).__init__(**kwargs)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=False)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=False)  # 把query和key映射到同一个隐藏空间
        self.w_v = nn.Linear(num_hiddens, 1, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, keys, values, valid_lens):
        queries, keys = self.W_q(queries), self.W_k(keys)
        # 在维度扩展后，
        # queries的形状：(batch_size，查询的个数，1，num_hidden)
        # key的形状：(batch_size，1，“键－值”对的个数，num_hiddens)
        # 使用广播方式进行求和
        features = queries.unsqueeze(2) + keys.unsqueeze(1)  # 在指定位置添加一个1维度，便于广播求和
        features = torch.tanh(features)  #(B, query数量, key数量, num_hiddens)
        # self.w_v仅有一个输出，因此从形状中移除最后那个维度。
        # scores的形状：(batch_size，查询的个数，“键-值”对的个数)
        scores = self.w_v(features).squeeze(-1)  # （B， Q， K， 1）即每个QK都会得到一个标量
        self.attention_weights = masked_softmax(scores, valid_lens)
        # values的形状：(batch_size，“键－值”对的个数，值的维度)
        return torch.bmm(self.dropout(self.attention_weights), values)  # (B, Q, K) * (B, K, V) -> (B, Q, V) 进行加权求和

添加一个AdditiveAttention类，测试上面的代码

In [ ]:
queries, keys = torch.normal(0, 1, (2, 1, 20)), torch.ones((2, 10, 2))  # 这里面，batch为2，每个样本有一个query，20维向量；每个样本有10个key，每个key是2维向量
# values的小批量，两个值矩阵是相同的
values = torch.arange(40, dtype=torch.float32).reshape(1, 10, 4).repeat(
    2, 1, 1)
valid_lens = torch.tensor([2, 6])

attention = AdditiveAttention(key_size=2, query_size=20, num_hiddens=8,
                              dropout=0.1)
attention.eval()
attention(queries, keys, values, valid_lens)